# EAD Model Demo — CreditRisk Intelligence
### Exposure at Default: Credit Conversion Factor (CCF) modelling

**Author:** Rajan Puri  
**Repo:** [github.com/purirajan/creditriskintelligence](https://github.com/purirajan/creditriskintelligence)

---

**Exposure at Default (EAD)** is the total value exposed to loss at the moment
of default. For term loans it is simply the outstanding balance. For **revolving
facilities** (credit cards, BNPL credit lines, HELOCs) it requires estimating
how much of the **undrawn commitment** the borrower will draw before defaulting:

> **EAD = Drawn Balance + CCF × (Credit Limit − Drawn Balance)**

The **Credit Conversion Factor (CCF)** ∈ [0, 1] is the fraction of undrawn
commitment expected to be drawn by time of default. Borrowers in distress
typically **draw down more** before defaulting — making CCF a critical and
often under-estimated risk driver.

### What this notebook covers

| Step | Description |
|---|---|
| 1. Data generation | Synthetic revolving facility dataset with realistic drawdown behaviour |
| 2. EDA | CCF distributions, drawdown by utilisation band, product analysis |
| 3. CCF model | Ridge regression + XGBoost for CCF estimation |
| 4. Regulatory floors | Basel III CCF floors by product type (CRR Art. 111) |
| 5. EAD calculation | EAD = drawn + CCF × undrawn, model vs regulatory |
| 6. Validation | MAE, calibration, stress testing |
| 7. SHAP explainability | Key drivers of drawdown behaviour |
| 8. Full ECL | PD × LGD × EAD complete pipeline preview |

### Regulatory alignment
- **Basel II/III IRB** — EAD pillar (CCF for revolving facilities)  
- **CECL (ASC 326)** — EAD for revolving credit commitments  
- **IFRS 9** — Stage 2/3 ECL exposure estimation  
- **SR 11-7** — Model governance documentation


## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import shap

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
})
TEAL  = '#1D9E75'
CORAL = '#D85A30'
BLUE  = '#378ADD'
AMBER = '#EF9F27'
PURPLE= '#7F77DD'

print("✅ Libraries loaded")
print(f"   numpy  {np.__version__}")
print(f"   pandas {pd.__version__}")
print(f"   xgb    {xgb.__version__}")


## 1. Data — Revolving Facility Drawdown Dataset

CCF models are trained on defaulted **revolving accounts** where we know:
1. The drawn balance and credit limit **at the observation date** (t₀)
2. The drawn balance **at the default date** (t_default)

From this we compute the realised CCF:
> **CCF_realised = (Drawn_at_default − Drawn_at_obs) / Undrawn_at_obs**

Key behavioural insight: borrowers in financial distress tend to **max out**
their credit lines before defaulting — the "credit card death spiral."


In [ ]:
def generate_ead_dataset(n: int = 20_000, seed: int = 42) -> pd.DataFrame:
    """
    Generate a realistic CCF dataset for revolving credit facilities.
    Captures the key behavioural insight: higher-risk borrowers draw more.
    """
    rng = np.random.default_rng(seed)

    product_code = rng.choice([0, 1, 2, 3],
                               p=[0.25, 0.30, 0.30, 0.15], size=n)
    product_name = np.array(['Personal revolving','Credit card','BNPL','HELOC'])[product_code]

    credit_limit       = np.clip(rng.lognormal(8.5, 0.8, n), 500, 100_000)
    utilization_rate   = np.clip(rng.beta(2, 3, n), 0.01, 0.99)
    current_balance    = credit_limit * utilization_rate
    undrawn            = credit_limit - current_balance

    months_to_maturity      = np.where(product_code == 3,
                                        rng.integers(60, 360, n).astype(float),
                                        0.0)   # revolving = no fixed maturity
    months_since_last_draw  = rng.integers(0, 36, n).astype(float)
    payment_behaviour_score = np.clip(rng.normal(65, 18, n), 0, 100)
    fico_proxy              = np.clip(rng.normal(640, 70, n), 300, 850)
    income                  = np.clip(rng.lognormal(10.8, 0.5, n), 10_000, 300_000)
    dti_ratio               = np.clip(rng.beta(2, 5, n), 0.01, 0.95)
    delinquency_count       = rng.poisson(0.5, n).astype(float)
    months_on_book          = rng.integers(1, 120, n).astype(float)

    # CCF is higher when:
    #  - borrower has high current utilization (already stressed)
    #  - low payment behaviour score (unreliable payer)
    #  - short months_since_last_draw (active user)
    #  - BNPL or credit card (easier to draw)
    logit_ccf = (
        -0.5
        + utilization_rate         * 3.0   # high util → more likely to max out
        - payment_behaviour_score  * 0.03  # poor payer → draws more
        - months_since_last_draw   * 0.04  # recent activity → more draw
        + (product_code == 1)      * 0.8   # credit card: easy tap-and-go
        + (product_code == 2)      * 0.5   # BNPL: impulse drawing
        - (product_code == 3)      * 0.6   # HELOC: harder to draw (secured)
        + dti_ratio                * 1.5   # high DTI = financial stress
        - (fico_proxy - 640)       * 0.005 # lower FICO → higher drawdown
        + delinquency_count        * 0.2   # more delinquencies → more draw
        + rng.normal(0, 0.5, n)
    )
    ccf_raw = 1 / (1 + np.exp(-logit_ccf))
    ccf     = np.clip(ccf_raw, 0.001, 0.999)

    # Compute EAD
    ead = current_balance + ccf * undrawn

    return pd.DataFrame({
        'ccf':                      ccf.round(5),
        'ead':                      ead.round(2),
        'current_balance':          current_balance.round(2),
        'credit_limit':             credit_limit.round(2),
        'undrawn_commitment':       undrawn.round(2),
        'utilization_rate':         utilization_rate.round(4),
        'months_to_maturity':       months_to_maturity,
        'product_type_code':        product_code,
        'product_name':             product_name,
        'months_since_last_draw':   months_since_last_draw,
        'payment_behaviour_score':  payment_behaviour_score.round(1),
        'fico_proxy':               fico_proxy.round(0),
        'income':                   income.round(0),
        'dti_ratio':                dti_ratio.round(4),
        'delinquency_count':        delinquency_count,
        'months_on_book':           months_on_book,
    })

df = generate_ead_dataset(n=20_000)

print(f"Dataset: {len(df):,} defaulted revolving accounts")
print()
print(df[['ccf','current_balance','credit_limit','utilization_rate',
           'payment_behaviour_score']].describe().round(3).to_string())


## 2. Exploratory Data Analysis

In [ ]:
# ── 2.1  CCF distributions ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Overall CCF histogram
axes[0].hist(df['ccf'], bins=60, color=BLUE, alpha=0.8, edgecolor='white')
axes[0].axvline(df['ccf'].mean(),   color='black',  linestyle='--',
                linewidth=1.5, label=f"Mean CCF = {df['ccf'].mean():.3f}")
axes[0].axvline(df['ccf'].median(), color=CORAL, linestyle=':',
                linewidth=1.5, label=f"Median CCF = {df['ccf'].median():.3f}")
axes[0].set_xlabel('CCF (Credit Conversion Factor)')
axes[0].set_ylabel('Count')
axes[0].set_title('CCF distribution (all products)', fontweight='500')
axes[0].legend()

# CCF by product
colors_prod = [TEAL, BLUE, AMBER, CORAL]
for i, (prod, grp) in enumerate(df.groupby('product_name')):
    axes[1].hist(grp['ccf'], bins=40, alpha=0.55,
                 color=colors_prod[i], label=prod, density=True)
axes[1].set_xlabel('CCF')
axes[1].set_ylabel('Density')
axes[1].set_title('CCF by product type', fontweight='500')
axes[1].legend(fontsize=9)

# CCF by utilisation band
df['util_band'] = pd.cut(df['utilization_rate'],
                          bins=[0, 0.25, 0.50, 0.75, 1.0],
                          labels=['0-25%','25-50%','50-75%','75-100%'])
ccf_by_util = df.groupby('util_band', observed=True)['ccf'].mean()
axes[2].bar(ccf_by_util.index, ccf_by_util.values,
            color=[TEAL, BLUE, AMBER, CORAL], edgecolor='white', width=0.6)
axes[2].set_xlabel('Current utilisation band')
axes[2].set_ylabel('Mean CCF')
axes[2].set_title('Mean CCF by utilisation band', fontweight='500')
for i, (band, val) in enumerate(ccf_by_util.items()):
    axes[2].text(i, val + 0.005, f'{val:.3f}',
                 ha='center', fontsize=10, fontweight='500')

plt.suptitle('Figure 1 — CCF distributions: higher utilisation → higher drawdown',
             fontsize=12, fontweight='500')
plt.tight_layout()
plt.show()

print("Key insight:")
print("  High-utilisation borrowers (75-100%) have the highest CCF →")
print("  they are most likely to max out before defaulting.")


In [ ]:
# ── 2.2  CCF vs key drivers (decile analysis) ────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

drivers = [
    ('utilization_rate',        'Utilisation rate'),
    ('payment_behaviour_score', 'Payment behaviour score'),
    ('months_since_last_draw',  'Months since last draw'),
    ('dti_ratio',               'DTI ratio'),
    ('fico_proxy',              'FICO score (proxy)'),
    ('delinquency_count',       'Delinquency count'),
]

for i, (col, label) in enumerate(drivers):
    df['_bin'] = pd.qcut(df[col].clip(
        lower=df[col].quantile(0.01), upper=df[col].quantile(0.99)
    ), q=10, duplicates='drop', labels=False)
    means = df.groupby('_bin')['ccf'].mean()

    axes[i].plot(range(len(means)), means.values, 'o-',
                 color=BLUE, linewidth=2, markersize=5)
    axes[i].axhline(df['ccf'].mean(), color='gray', linestyle='--',
                    linewidth=1, alpha=0.7, label=f'Avg {df["ccf"].mean():.3f}')
    axes[i].set_xlabel(f'{label} decile')
    axes[i].set_ylabel('Mean CCF')
    axes[i].set_title(label, fontweight='500')
    axes[i].legend(fontsize=9)

df.drop(columns=['_bin'], inplace=True, errors='ignore')

plt.suptitle('Figure 2 — Mean CCF by feature decile',
             fontsize=12, fontweight='500')
plt.tight_layout()
plt.show()


In [ ]:
# ── 2.3  EAD vs current balance ───────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# EAD vs balance scatter
sample = df.sample(3000, random_state=1)
sc = axes[0].scatter(sample['current_balance'], sample['ead'],
                     c=sample['ccf'], cmap='YlOrRd',
                     alpha=0.5, s=8)
axes[0].plot([0, df['credit_limit'].max()],
             [0, df['credit_limit'].max()],
             'gray', linestyle='--', linewidth=1, label='EAD = balance (CCF=0)')
axes[0].set_xlabel('Current drawn balance ($)')
axes[0].set_ylabel('EAD ($)')
axes[0].set_title('EAD vs current balance (colour=CCF)', fontweight='500')
plt.colorbar(sc, ax=axes[0], label='CCF')
axes[0].legend()

# EAD uplift (EAD - balance) = CCF × undrawn
df['ead_uplift'] = df['ead'] - df['current_balance']
axes[1].hist(df['ead_uplift'].clip(upper=df['ead_uplift'].quantile(0.99)),
             bins=60, color=CORAL, alpha=0.8, edgecolor='white')
axes[1].axvline(df['ead_uplift'].mean(), color='black', linestyle='--',
                linewidth=1.5, label=f"Mean uplift = ${df['ead_uplift'].mean():,.0f}")
axes[1].set_xlabel('EAD uplift = CCF × undrawn ($)')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of EAD uplift from CCF', fontweight='500')
axes[1].legend()

plt.suptitle('Figure 3 — EAD uplift: how much more exposure exists beyond current balance',
             fontsize=12, fontweight='500')
plt.tight_layout()
plt.show()

print(f"Mean EAD uplift   : ${df['ead_uplift'].mean():,.2f}")
print(f"Median EAD uplift : ${df['ead_uplift'].median():,.2f}")
print(f"Total portfolio EAD uplift: ${df['ead_uplift'].sum():,.0f}")
print(f"  (this is the exposure that balance sheet alone would miss)")


## 3. Feature Preparation

In [ ]:
FEATURES = [
    'current_balance',
    'credit_limit',
    'utilization_rate',
    'months_to_maturity',
    'product_type_code',
    'months_since_last_draw',
    'payment_behaviour_score',
    'dti_ratio',
    'delinquency_count',
    'months_on_book',
]

TARGET = 'ccf'

X = df[FEATURES].copy()
y = np.clip(df[TARGET], 0.001, 0.999)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print(f"Train : {len(X_train):,} rows  |  mean CCF: {y_train.mean():.4f}")
print(f"Test  : {len(X_test):,} rows  |  mean CCF: {y_test.mean():.4f}")
print(f"Features used: {FEATURES}")


## 4. Model A — Ridge Regression (Regulatory Baseline)

Ridge regression provides an **interpretable linear baseline** for the CCF model.
Regulators expect a clearly documented linear model as the primary model for
Basel IRB submissions — XGBoost serves as a challenger/operational model.


In [ ]:
# ── 4.1  Ridge regression ─────────────────────────────────────────────────
ridge_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('reg',    Ridge(alpha=1.0)),
])
ridge_pipeline.fit(X_train, y_train)

ridge_preds = np.clip(ridge_pipeline.predict(X_test), 0.001, 0.999)
ridge_mae   = mean_absolute_error(y_test, ridge_preds)
ridge_rmse  = np.sqrt(mean_squared_error(y_test, ridge_preds))
ridge_r2    = r2_score(y_test, ridge_preds)

print("Ridge Regression — Test performance:")
print(f"  MAE  : {ridge_mae:.5f}")
print(f"  RMSE : {ridge_rmse:.5f}")
print(f"  R²   : {ridge_r2:.4f}")


In [ ]:
# ── 4.2  Coefficient table ────────────────────────────────────────────────
coefs = ridge_pipeline.named_steps['reg'].coef_
scale = ridge_pipeline.named_steps['scaler'].scale_
std_coefs = coefs * scale

coef_df = pd.DataFrame({
    'feature':   FEATURES,
    'raw_coef':  coefs.round(5),
    'std_coef':  std_coefs.round(5),
    'direction': ['↑ CCF' if c > 0 else '↓ CCF' for c in coefs],
}).sort_values('std_coef', key=abs, ascending=False)

print("Ridge CCF model — standardised coefficients:")
print(coef_df.to_string(index=False))
print()
print("→ utilization_rate (+): higher current draw → more likely to max out")
print("→ payment_behaviour_score (-): reliable payer → less drawdown")
print("→ months_since_last_draw (-): inactive account → less drawdown")


## 5. Model B — XGBoost CCF Model

In [ ]:
# ── 5.1  Train XGBoost ────────────────────────────────────────────────────
X_tr, X_va, y_tr, y_va = train_test_split(
    X_train, y_train, test_size=0.15, random_state=0
)

xgb_ccf = xgb.XGBRegressor(
    n_estimators=600,
    max_depth=4,
    learning_rate=0.02,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=10,
    gamma=0.05,
    reg_alpha=0.1,
    reg_lambda=1.5,
    objective='reg:squarederror',
    eval_metric='mae',
    early_stopping_rounds=30,
    random_state=42,
    verbosity=0,
)
xgb_ccf.fit(
    X_tr, y_tr,
    eval_set=[(X_va, y_va)],
    verbose=False,
)

xgb_preds = np.clip(xgb_ccf.predict(X_test), 0.001, 0.999)
xgb_mae   = mean_absolute_error(y_test, xgb_preds)
xgb_rmse  = np.sqrt(mean_squared_error(y_test, xgb_preds))
xgb_r2    = r2_score(y_test, xgb_preds)

print("XGBoost CCF — Test performance:")
print(f"  MAE  : {xgb_mae:.5f}")
print(f"  RMSE : {xgb_rmse:.5f}")
print(f"  R²   : {xgb_r2:.4f}")
print(f"  Best iteration: {xgb_ccf.best_iteration}")
print()
print("Model comparison:")
print(f"  {'Model':<28} {'MAE':>8} {'RMSE':>8} {'R²':>8}")
print(f"  {'-'*52}")
print(f"  {'Ridge (Regulatory baseline)':<28} {ridge_mae:>8.5f} {ridge_rmse:>8.5f} {ridge_r2:>8.4f}")
print(f"  {'XGBoost (Operational)':<28} {xgb_mae:>8.5f} {xgb_rmse:>8.5f} {xgb_r2:>8.4f}")


## 6. Basel III Regulatory CCF Floors

Basel III CRR Article 111 defines **minimum CCF floors** for revolving exposures.
The model CCF must be **floored** at these values — we always take the higher of
model estimate and regulatory minimum.

| Product | Regulatory CCF floor | Rationale |
|---|---|---|
| Personal revolving | 75% | Unsecured, high drawdown risk |
| Credit card | 75% | Easy tap-and-draw, behavioural evidence |
| BNPL | 50% | Shorter tenors, lower drawdown uncertainty |
| HELOC | 50% | Secured, slower drawdown process |


In [ ]:
# ── 6.1  Apply regulatory CCF floors ─────────────────────────────────────
REG_CCF_FLOORS = {
    0: 0.75,   # personal revolving
    1: 0.75,   # credit card
    2: 0.50,   # BNPL
    3: 0.50,   # HELOC
}

def apply_regulatory_floor(ccf_model: np.ndarray,
                             product_codes: np.ndarray) -> np.ndarray:
    """Apply Basel III CCF floors: max(model_ccf, regulatory_floor)."""
    floors = np.array([REG_CCF_FLOORS[int(p)] for p in product_codes])
    return np.maximum(ccf_model, floors)

ccf_model_test = xgb_preds
ccf_reg_test   = apply_regulatory_floor(
    ccf_model_test, X_test['product_type_code'].values
)

product_names = {0:'Personal revolving',1:'Credit card',2:'BNPL',3:'HELOC'}
print("Model CCF vs Regulatory CCF (with floors) by product:")
print(f"  {'Product':<22} {'Reg floor':>10} {'Model CCF':>10} {'Final CCF':>10} {'Floor binding?':>15}")
print(f"  {'-'*70}")
for code, name in product_names.items():
    mask   = X_test['product_type_code'].values == code
    floor  = REG_CCF_FLOORS[code]
    m_ccf  = ccf_model_test[mask].mean()
    f_ccf  = ccf_reg_test[mask].mean()
    binding = '✅ binding' if f_ccf > m_ccf else '— model higher'
    print(f"  {name:<22} {floor:>10.2f} {m_ccf:>10.4f} {f_ccf:>10.4f} {binding:>15}")


In [ ]:
# ── 6.2  CCF floor impact visualisation ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: model vs regulatory CCF
axes[0].scatter(ccf_model_test, ccf_reg_test, alpha=0.3, s=6,
                c=X_test['product_type_code'].values, cmap='tab10')
axes[0].plot([0,1],[0,1], 'gray', linestyle='--', linewidth=1, label='No floor impact')
axes[0].axhline(0.75, color=CORAL, linestyle=':', linewidth=1, alpha=0.6, label='75% floor')
axes[0].axhline(0.50, color=AMBER, linestyle=':', linewidth=1, alpha=0.6, label='50% floor')
axes[0].set_xlabel('Model CCF')
axes[0].set_ylabel('Regulatory CCF (with floor)')
axes[0].set_title('Model vs regulatory CCF', fontweight='500')
axes[0].legend(fontsize=9)

# % of accounts where floor is binding by product
pct_floored = {}
for code, name in product_names.items():
    mask = X_test['product_type_code'].values == code
    floored = (ccf_reg_test[mask] > ccf_model_test[mask]).mean()
    pct_floored[name] = floored * 100

axes[1].bar(pct_floored.keys(), pct_floored.values(),
            color=[TEAL, BLUE, AMBER, CORAL], edgecolor='white', width=0.6)
axes[1].set_ylabel('% accounts where floor is binding')
axes[1].set_title('Regulatory floor binding rate by product', fontweight='500')
for i, (name, pct) in enumerate(pct_floored.items()):
    axes[1].text(i, pct + 0.5, f'{pct:.1f}%', ha='center', fontsize=10, fontweight='500')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('Figure 4 — Regulatory CCF floor impact: Basel III compliance',
             fontsize=12, fontweight='500')
plt.tight_layout()
plt.show()


## 7. EAD Calculation

In [ ]:
# ── 7.1  Compute EAD from CCF ─────────────────────────────────────────────
balance  = X_test['current_balance'].values
limit    = X_test['credit_limit'].values
undrawn  = np.maximum(limit - balance, 0)

ead_model       = balance + ccf_model_test  * undrawn
ead_regulatory  = balance + ccf_reg_test    * undrawn
ead_min_floor   = balance                              # CCF = 0 (no drawdown)
ead_max_floor   = limit                                # CCF = 1 (full limit drawn)

print("EAD summary across test portfolio:")
print(f"  {'Scenario':<30} {'Mean EAD':>12} {'Total EAD':>16}")
print(f"  {'-'*60}")
print(f"  {'Balance only (no CCF)':<30} ${balance.mean():>11,.2f}  ${balance.sum():>15,.0f}")
print(f"  {'Model EAD':<30} ${ead_model.mean():>11,.2f}  ${ead_model.sum():>15,.0f}")
print(f"  {'Regulatory EAD (with floors)':<30} ${ead_regulatory.mean():>11,.2f}  ${ead_regulatory.sum():>15,.0f}")
print(f"  {'Maximum EAD (CCF=1)':<30} ${ead_max_floor.mean():>11,.2f}  ${ead_max_floor.sum():>15,.0f}")
print()
print(f"  EAD uplift (model vs balance): +${(ead_model-balance).mean():,.2f} per account")
print(f"  EAD uplift (reg vs balance):   +${(ead_regulatory-balance).mean():,.2f} per account")
print()
print("  → Regulatory EAD is always ≥ model EAD due to floors")
print("  → Using balance only UNDER-ESTIMATES true exposure")


In [ ]:
# ── 7.2  EAD comparison visualisation ────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# EAD scenario comparison
scenarios = {
    'Balance\nonly': balance,
    'Model\nEAD':    ead_model,
    'Reg EAD\n(floors)': ead_regulatory,
    'Max EAD\n(CCF=1)': ead_max_floor,
}
colors_s = [TEAL, BLUE, CORAL, AMBER]
means    = [v.mean() for v in scenarios.values()]
bars = axes[0].bar(scenarios.keys(), means, color=colors_s,
                   edgecolor='white', width=0.6)
axes[0].set_ylabel('Mean EAD per account ($)')
axes[0].set_title('EAD by scenario', fontweight='500')
for bar, val in zip(bars, means):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 50,
                 f'${val:,.0f}', ha='center', fontsize=9, fontweight='500')

# EAD uplift distribution by product
ead_uplift = ead_regulatory - balance
uplift_by_prod = {}
for code, name in product_names.items():
    mask = X_test['product_type_code'].values == code
    uplift_by_prod[name] = ead_uplift[mask]

axes[1].boxplot(uplift_by_prod.values(), labels=uplift_by_prod.keys(),
                patch_artist=True,
                boxprops=dict(facecolor=BLUE, alpha=0.5),
                medianprops=dict(color=CORAL, linewidth=2))
axes[1].set_ylabel('EAD uplift from CCF ($)')
axes[1].set_title('EAD uplift distribution by product', fontweight='500')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('Figure 5 — EAD scenarios and uplift analysis',
             fontsize=12, fontweight='500')
plt.tight_layout()
plt.show()


## 8. Model Validation

In [ ]:
# ── 8.1  CCF calibration by decile ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, preds, title, color in [
    (axes[0], ridge_preds, 'Ridge Regression', BLUE),
    (axes[1], xgb_preds,   'XGBoost',          TEAL),
]:
    cal_df = pd.DataFrame({'actual': y_test.values, 'predicted': preds})
    cal_df['decile'] = pd.qcut(cal_df['predicted'], q=10,
                                labels=False, duplicates='drop')
    cal_agg = cal_df.groupby('decile').agg(
        mean_actual=('actual',    'mean'),
        mean_pred  =('predicted', 'mean'),
        count      =('actual',    'count'),
    )

    ax.scatter(cal_agg['mean_pred'], cal_agg['mean_actual'],
               s=cal_agg['count'] / 8, color=color, alpha=0.8, zorder=5)
    ax.plot([0,1],[0,1], 'gray', linestyle='--', linewidth=1.5,
            label='Perfect calibration')

    for _, row in cal_agg.iterrows():
        ax.annotate(f"{row['count']:.0f}",
                    (row['mean_pred'], row['mean_actual']),
                    textcoords='offset points', xytext=(4,3), fontsize=8)

    ax.set_xlabel('Mean predicted CCF (decile)')
    ax.set_ylabel('Mean actual CCF (decile)')
    ax.set_title(f'CCF calibration — {title}', fontweight='500')
    ax.legend()

plt.suptitle('Figure 6 — CCF calibration: predicted vs actual by decile',
             fontsize=12, fontweight='500')
plt.tight_layout()
plt.show()


In [ ]:
# ── 8.2  Cross-validation stability ──────────────────────────────────────
print("5-fold cross-validation (XGBoost CCF)...")

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_maes = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(X_train), 1):
    X_tr_cv, X_va_cv = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    y_tr_cv, y_va_cv = y_train.iloc[tr_idx], y_train.iloc[va_idx]

    m = xgb.XGBRegressor(
        n_estimators=xgb_ccf.best_iteration,
        max_depth=4, learning_rate=0.02,
        subsample=0.8, colsample_bytree=0.8,
        min_child_weight=10, random_state=42, verbosity=0,
    )
    m.fit(X_tr_cv, y_tr_cv)
    mae_cv = mean_absolute_error(y_va_cv, np.clip(m.predict(X_va_cv), 0, 1))
    cv_maes.append(mae_cv)
    print(f"  Fold {fold}: MAE = {mae_cv:.5f}")

print(f"  ─────────────────────────")
print(f"  Mean MAE : {np.mean(cv_maes):.5f}")
print(f"  Std  MAE : {np.std(cv_maes):.5f}  (low = stable ✅)")


## 9. SHAP Explainability

In [ ]:
# ── 9.1  TreeExplainer ───────────────────────────────────────────────────
print("Computing SHAP values...")
explainer = shap.TreeExplainer(xgb_ccf)
shap_vals = explainer.shap_values(X_test)
print(f"✅ SHAP values: {shap_vals.shape}")


In [ ]:
# ── 9.2  Global importance + beeswarm ────────────────────────────────────
mean_abs  = np.abs(shap_vals).mean(axis=0)
shap_imp  = pd.Series(mean_abs, index=FEATURES).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Bar
colors_bar = [CORAL if v == shap_imp.max() else TEAL for v in shap_imp.values]
bars = axes[0].barh(shap_imp.index, shap_imp.values,
                    color=colors_bar, height=0.6, edgecolor='white')
for bar, val in zip(bars, shap_imp.values):
    axes[0].text(val + 0.0001, bar.get_y() + bar.get_height()/2,
                 f'{val:.4f}', va='center', fontsize=9)
axes[0].set_xlabel('Mean |SHAP value|')
axes[0].set_title('Figure 7 — CCF feature importance (mean |SHAP|)', fontweight='500')

# Beeswarm
shap_exp = shap.Explanation(
    values=shap_vals,
    base_values=explainer.expected_value,
    data=X_test.values,
    feature_names=FEATURES,
)
plt.sca(axes[1])
shap.plots.beeswarm(shap_exp, max_display=8, show=False)
axes[1].set_title('SHAP beeswarm — CCF drivers', fontweight='500')

plt.tight_layout()
plt.show()

print("\nTop CCF drivers:")
for rank, (feat, val) in enumerate(shap_imp.sort_values(ascending=False).items(), 1):
    print(f"  {rank}. {feat:<28} {val:.5f}")


In [ ]:
# ── 9.3  Waterfall for highest CCF account (max drawdown risk) ───────────
worst_ccf_idx = np.argmax(xgb_preds)

single_exp = shap.Explanation(
    values=shap_vals[worst_ccf_idx],
    base_values=explainer.expected_value,
    data=X_test.iloc[worst_ccf_idx].values,
    feature_names=FEATURES,
)

plt.figure(figsize=(10, 4))
shap.plots.waterfall(single_exp, show=False)
plt.title(f'Figure 8 — Waterfall: highest drawdown risk '
          f'(CCF = {xgb_preds[worst_ccf_idx]:.2%})',
          fontweight='500', pad=12)
plt.tight_layout()
plt.show()

acct = X_test.iloc[worst_ccf_idx]
print(f"Highest drawdown risk account:")
for f in FEATURES:
    print(f"  {f:<28} {acct[f]:.3f}")
print(f"  Model CCF       : {xgb_preds[worst_ccf_idx]:.2%}")
print(f"  Regulatory CCF  : {ccf_reg_test[worst_ccf_idx]:.2%}")
print(f"  EAD (model)     : ${ead_model[worst_ccf_idx]:,.2f}")
print(f"  EAD (regulatory): ${ead_regulatory[worst_ccf_idx]:,.2f}")


## 10. Full ECL Preview — PD × LGD × EAD

In [ ]:
# ── 10.1  Combine all three components ───────────────────────────────────
rng = np.random.default_rng(7)

# PD proxy (use PDModel in production)
pd_proxy = np.clip(
    0.07
    + (1 - X_test['payment_behaviour_score'].values / 100) * 0.12
    + X_test['utilization_rate'].values * 0.08
    + X_test['dti_ratio'].values * 0.06
    + X_test['delinquency_count'].values * 0.03
    + rng.normal(0, 0.01, len(X_test)),
    0.001, 0.999
)

# LGD proxy (use LGDModel in production — unsecured = higher LGD)
lgd_proxy = np.clip(
    0.55
    - X_test['utilization_rate'].values * 0.1
    + rng.normal(0, 0.03, len(X_test)),
    0.20, 0.90
)

ead_final = ead_regulatory   # use Basel III regulatory EAD

ecl_12m   = pd_proxy * lgd_proxy * ead_final

print("Full ECL = PD × LGD × EAD  (test portfolio):")
print(f"  {'Component':<30} {'Mean':>12} {'Portfolio total':>18}")
print(f"  {'-'*62}")
print(f"  {'PD (proxy)':<30} {pd_proxy.mean():>11.3%}")
print(f"  {'LGD (proxy)':<30} {lgd_proxy.mean():>11.3%}")
print(f"  {'EAD — balance only':<30} ${balance.mean():>11,.2f}  ${balance.sum():>17,.0f}")
print(f"  {'EAD — regulatory (with CCF)':<30} ${ead_final.mean():>11,.2f}  ${ead_final.sum():>17,.0f}")
print()
print(f"  {'ECL using balance only':<30} ${(pd_proxy*lgd_proxy*balance).mean():>11,.2f}  ${(pd_proxy*lgd_proxy*balance).sum():>17,.0f}")
print(f"  {'ECL using regulatory EAD':<30} ${ecl_12m.mean():>11,.2f}  ${ecl_12m.sum():>17,.0f}")
print()
uplift_pct = (ecl_12m.sum()/(pd_proxy*lgd_proxy*balance).sum()-1)
print(f"  ECL uplift from CCF modelling: +{uplift_pct:.1%}")
print()
print("  → Ignoring CCF UNDER-REPORTS true expected credit loss")


In [ ]:
# ── 10.2  ECL decomposition ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ECL distribution
axes[0].hist(ecl_12m.clip(upper=np.percentile(ecl_12m, 98)),
             bins=60, color=CORAL, alpha=0.8, edgecolor='white')
axes[0].axvline(ecl_12m.mean(), color='black', linestyle='--',
                linewidth=1.5, label=f'Mean ECL = ${ecl_12m.mean():,.0f}')
axes[0].set_xlabel('12-month ECL per account ($)')
axes[0].set_ylabel('Count')
axes[0].set_title('ECL distribution (regulatory EAD)', fontweight='500')
axes[0].legend()

# Scatter: PD vs ECL, coloured by LGD
sc = axes[1].scatter(pd_proxy, ecl_12m, c=lgd_proxy,
                     cmap='YlOrRd', alpha=0.3, s=6)
axes[1].set_xlabel('PD')
axes[1].set_ylabel('ECL ($)')
axes[1].set_title('ECL = PD × LGD × EAD  (colour = LGD)', fontweight='500')
plt.colorbar(sc, ax=axes[1], label='LGD')

plt.suptitle('Figure 9 — Full ECL: PD × LGD × EAD with regulatory CCF',
             fontsize=12, fontweight='500')
plt.tight_layout()
plt.show()


## 11. Summary

In [ ]:
print("=" * 65)
print("  MODEL CARD — CreditRisk Intelligence EAD / CCF Model Demo")
print("=" * 65)
print(f"  Dataset       : Synthetic revolving facility (20k accounts)")
print(f"  Target        : CCF ∈ [0, 1]  (fraction of undrawn drawn at default)")
print(f"  Features      : {len(FEATURES)}")
print(f"  Mean CCF      : {y.mean():.3f}")
print()
print("  ── Ridge Regression (Regulatory baseline) ──────────────────")
print(f"  MAE  : {ridge_mae:.5f}   RMSE: {ridge_rmse:.5f}   R²: {ridge_r2:.4f}")
print(f"  Use  : Audit trail, regulatory submission, interpretable")
print()
print("  ── XGBoost (Operational scoring) ────────────────────────────")
print(f"  MAE  : {xgb_mae:.5f}   RMSE: {xgb_rmse:.5f}   R²: {xgb_r2:.4f}")
print(f"  Use  : Daily EAD pipeline, portfolio stress testing")
print()
print("  ── EAD uplift (regulatory CCF vs balance sheet) ─────────────")
print(f"  Mean EAD uplift  : +${(ead_regulatory-balance).mean():,.2f} per account")
print(f"  ECL uplift       : +{uplift_pct:.1%} vs balance-only ECL")
print()
print("  ── Regulatory alignment ─────────────────────────────────────")
print("  Basel II/III IRB — EAD pillar, CCF floors (CRR Art. 111)")
print("  CECL (ASC 326)   — Revolving credit EAD")
print("  IFRS 9           — Stage 2/3 ECL exposure")
print("  SR 11-7          — Model governance documentation")
print()
print("  ── Full ECL (PD × LGD × EAD) ───────────────────────────────")
print(f"  Mean 12-month ECL : ${ecl_12m.mean():,.2f} per account")
print(f"  Total ECL         : ${ecl_12m.sum():,.0f}")
print()
print("  Next: 03_ecl_full_pipeline.ipynb — integrated ECL with")
print("        vintage curves, stress scenarios, and CECL allowance")
print("=" * 65)


---

## Using the EAD model in the production API

```python
from src.models.ead_model import EADModel

model = EADModel(apply_regulatory_floors=True)
model.fit(X_train, y_train)

result = model.predict({
    'current_balance':          1200.0,
    'credit_limit':             5000.0,
    'utilization_rate':         0.24,
    'months_to_maturity':       0,
    'product_type_code':        1,   # credit card
    'months_since_last_draw':   2,
    'payment_behaviour_score':  72.0,
    'dti_ratio':                0.38,
    'delinquency_count':        1.0,
    'months_on_book':           24.0,
}, application_id='APP-001')

print(result['ccf_estimate'])      # model CCF
print(result['ccf_with_floor'])    # Basel III floored CCF
print(result['ead_estimate'])      # model EAD ($)
print(result['ead_regulatory'])    # regulatory EAD ($)
print(result['undrawn_commitment'])# credit limit - balance
```

Or via the API:
```bash
curl -X POST http://localhost:8000/v1/score/ecl \
  -H "Content-Type: application/json" \
  -d '{
    "pd_inputs": {...},
    "lgd_inputs": {...},
    "ead_inputs": {
      "current_balance": 1200,
      "credit_limit": 5000,
      "utilization_rate": 0.24,
      "months_to_maturity": 0,
      "product_type_code": 1,
      "months_since_last_draw": 2,
      "payment_behaviour_score": 72
    }
  }'
```

**Built by [Rajan Puri](https://rajanpuri.com)** 
